# WAXAL-NET Zero-Shot Evaluation: All 19 Languages
Runs zero-shot evaluation for all 19 WAXAL languages across 3 models:
- **MMS-1B**: all 19 languages
- **Omnilingual-1B**: all 19 languages (slow with batch_size=1)
- **Whisper Large-v3**: all 19 languages (returns N/A for unsupported languages)

Total: 57 evaluation runs. Results are written to `/kaggle/working/outputs/`.

In [ ]:
import subprocess
subprocess.run([
    "pip", "install", "--quiet",
    "torch==2.8.0", "torchaudio==2.8.0",
    "accelerate==1.10.1", "transformers==4.51.3",
    "huggingface_hub==0.32.6", "datasets[audio]==3.6.0",
    "numpy==1.26.4", "jiwer==4.0.0", "omnilingual-asr==0.1.0",
    "PyYAML==6.0.3", "soundfile==0.13.1", "wandb==0.19.11"
], check=True)
print("Dependencies installed.")

In [ ]:
import os
os.environ["HF_HOME"] = "/kaggle/working/hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "/kaggle/working/hf_cache"
os.environ["WANDB_DISABLED"] = "true"
os.makedirs("/kaggle/working/hf_cache", exist_ok=True)
os.makedirs("/kaggle/working/outputs", exist_ok=True)
print("Environment configured.")

In [ ]:
import sys
sys.path.insert(0, "/kaggle/input/waxal-asr-code/asr_waxal")
print("Repository path added.")

In [ ]:
import subprocess, sys
LANGUAGES_MMS1B = [
    "acholi", "akan", "amharic", "dagaare", "dagbani", "ewe", "fula", "ikposo",
    "lingala", "luganda", "malagasy", "masaaba", "nyankole", "oromo", "shona",
    "sidama", "soga", "tigrinya", "wolaytta"
]
for lang in LANGUAGES_MMS1B:
    print(f"Running MMS-1B on {lang}...")
    result = subprocess.run(
        [sys.executable, "-m", "src.zero_shot_eval",
         "--config", f"configs/experiment/zero_shot_{lang}_mms_1b.yaml"],
        cwd="/kaggle/input/waxal-asr-code/asr_waxal",
        capture_output=True, text=True
    )
    last_lines = result.stdout.strip().split("\n")[-3:]
    print("  " + "\n  ".join(last_lines))
    if result.returncode != 0:
        print(f"  ERROR: {result.stderr[-500:]}")
print("MMS-1B done.")

In [ ]:
import subprocess, sys
LANGUAGES_OMNI = [
    "acholi", "akan", "amharic", "dagaare", "dagbani", "ewe", "fula", "ikposo",
    "lingala", "luganda", "malagasy", "masaaba", "nyankole", "oromo", "shona",
    "sidama", "soga", "tigrinya", "wolaytta"
]
for lang in LANGUAGES_OMNI:
    print(f"Running Omnilingual-1B on {lang}...")
    result = subprocess.run(
        [sys.executable, "-m", "src.zero_shot_eval",
         "--config", f"configs/experiment/zero_shot_{lang}_omnilingual_1b.yaml"],
        cwd="/kaggle/input/waxal-asr-code/asr_waxal",
        capture_output=True, text=True
    )
    last_lines = result.stdout.strip().split("\n")[-3:]
    print("  " + "\n  ".join(last_lines))
    if result.returncode != 0:
        print(f"  ERROR: {result.stderr[-500:]}")
print("Omnilingual-1B done.")

In [ ]:
import subprocess, sys
LANGUAGES_WHISPER = [
    "acholi", "akan", "amharic", "dagaare", "dagbani", "ewe", "fula", "ikposo",
    "lingala", "luganda", "malagasy", "masaaba", "nyankole", "oromo", "shona",
    "sidama", "soga", "tigrinya", "wolaytta"
]
# Note: Whisper Large-v3 only supports amharic, lingala, malagasy, shona.
# The other 15 languages will return N/A status quickly.
for lang in LANGUAGES_WHISPER:
    print(f"Running Whisper Large-v3 on {lang}...")
    result = subprocess.run(
        [sys.executable, "-m", "src.zero_shot_eval",
         "--config", f"configs/experiment/zero_shot_{lang}_whisper_large_v3.yaml"],
        cwd="/kaggle/input/waxal-asr-code/asr_waxal",
        capture_output=True, text=True
    )
    last_lines = result.stdout.strip().split("\n")[-3:]
    print("  " + "\n  ".join(last_lines))
    if result.returncode != 0:
        print(f"  ERROR: {result.stderr[-500:]}")
print("Whisper Large-v3 done.")

In [ ]:
import json
from pathlib import Path
LANGUAGES = ["acholi", "akan", "amharic", "dagaare", "dagbani", "ewe", "fula", "ikposo",
             "lingala", "luganda", "malagasy", "masaaba", "nyankole", "oromo", "shona",
             "sidama", "soga", "tigrinya", "wolaytta"]
MODELS = ["mms_1b", "omnilingual_1b", "whisper_large_v3"]
print(f"{'Language':<12} {'MMS-1B':>12} {'Omni-1B':>12} {'Whisp-Lv3':>12}")
print("-" * 52)
for lang in LANGUAGES:
    row = [lang]
    for model in MODELS:
        p = Path(f"/kaggle/working/outputs/zero_shot_{lang}_{model}/metrics.json")
        if p.exists():
            m = json.loads(p.read_text())
            if m.get("status") == "N/A":
                row.append("N/A")
            else:
                row.append(f"{m['wer_percent']:.1f}%")
        else:
            row.append("\u2014")
    print(f"{row[0]:<12} {row[1]:>12} {row[2]:>12} {row[3]:>12}")